# ESP gas-lock classification

Solution edition. Use the Day 3 deck and instructor guide.

## Learning outcomes
- Audit the schema and evaluation question.
- Compare a baseline and a small neural model using identical partitions.
- Explain errors and limitations using reproducible evidence.

## Dataset contract
Use the six measured numeric inputs. ENTITY defines validation groups; TimeSlot is metadata. The task classifies current labelled observations and does not establish advance warning. Training positives are about 38% of rows, not a presumed rare-event rate.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "curriculum" / "capstone.py").exists())
sys.path.insert(0, str(ROOT / "curriculum"))
from capstone import prepare, run, SPECS


## 1. Inspect the supplied training file
Only the training file is read here. Do not open final labels during model selection.

In [ ]:
project = "esp"
spec = SPECS[project]
path = ROOT / "neural_networks_course" / "Day_3_Capstone" / "exercises" / spec["folder"]
df = pd.read_csv(path / spec["train"])
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
df.head()

## 2. Write the prediction contract
State the target, row unit, available predictors, excluded metadata, evaluation population, and known missing metadata. Explain what the final score cannot establish.

Use the six measured numeric inputs. ENTITY defines validation groups; TimeSlot is metadata. The task classifies current labelled observations and does not establish advance warning. Training positives are about 38% of rows, not a presumed rare-event rate.

## 3. Inspect the split and fitted preprocessing
The reference implementation splits before fitting imputation and scaling. Inspect `curriculum/capstone.py`, especially `prepare`, before running it.

In [ ]:
data = prepare(project, seed=42)
print(data["split_policy"])
print("Training shape:", data["Xt"].shape)
print("Validation shape:", data["Xv"].shape)
assert not set(data["train_idx"]) & set(data["val_idx"])
assert np.isfinite(data["Xt"]).all()
assert np.isfinite(data["Xv"]).all()

## 4. Verify a preprocessing invariant
Check that the imputer medians equal medians of training rows only. Explain why using all rows would invalidate this check. For ESP, also check disjoint entity IDs.

In [ ]:
expected = data["X"].iloc[data["train_idx"]].median().to_numpy()
observed = data["prep"].named_steps["simpleimputer"].statistics_
np.testing.assert_allclose(observed, expected)
if project == "esp":
    train_ids = set(data["frame"].ENTITY.iloc[data["train_idx"]])
    val_ids = set(data["frame"].ENTITY.iloc[data["val_idx"]])
    assert train_ids.isdisjoint(val_ids)

## 5. Run one reproducible comparison
The reference run fits a linear baseline and a small dense network. It copies the best validation checkpoint, chooses a classification threshold on validation when applicable, saves artifacts, and checks prediction equivalence after reload. This is a starting experiment, not a tuned performance claim.

In [ ]:
run_dir = ROOT / "output" / "capstone" / "esp_solution"
result = run(project, epochs=40, width=32, lr=0.001, seed=42,
             final=False, out=run_dir)
pd.DataFrame(result["validation"]).T

## 6. Read the learning curves
Identify the selected epoch. Explain whether more training is likely to help. For regression, distinguish standardized training loss from original-target-unit evaluation metrics.

In [ ]:
import matplotlib.pyplot as plt
history = pd.read_csv(run_dir / "learning_curve.csv")
ax = history.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(9, 4))
ax.set_ylabel("Training objective")
ax.axvline(result["best_epoch"], linestyle="--", color="black")
plt.show()

## 7. Controlled experiment
Change one setting, such as width or learning rate, and write your hypothesis first. Run both settings with the same three or more seeds and compare the difference with the seed spread. `run()` keeps the validation split fixed (`split_seed=42` by default) whatever training seed you pass, so every run is scored on the same validation rows; compare the settings seed by seed. Compare validation results and runtime, and select the setting before opening final labels.

**Instructor guidance:** A useful change may worsen the score. Grade the hypothesis, controlled comparison, and explanation. Validation results are optimistic after model or threshold selection; final assessment remains separate.

## 8. Locked final assessment
Run only after selecting the complete configuration. The ESP and sensor final labels join by uuid with one-to-one validation. The PE file aligns positionally after a strict row-count check. Do not tune further on these scores.

In [ ]:
selected = result  # Set to the result of your chosen development run.
RUN_FINAL = False
if RUN_FINAL:
    final_result = run(project, epochs=selected["epochs_requested"],
                       width=selected["width"], lr=selected["learning_rate"],
                       seed=selected["seed"], final=True,
                       out=run_dir / "locked_final")
    print(json.dumps(final_result["final_test"], indent=2))

## 9. Error analysis and presentation
Use saved predictions to inspect one class, entity, interval, or input regime with weak performance. Prepare five slides: problem, model comparison, evidence plot, failure analysis, and recommendation. Include the limitations in the dataset contract.

## Submission
Submit this notebook after a fresh-kernel run, the selected configuration, split.csv, results.json, and the five-slide report. Do not claim a neural model is superior without the matched evaluation evidence.